# 03 — Palette (Conditional Diffusion) on Google Colab

**Project**: Black-and-White Image Colorization with Deep Learning (Proposal rev01, Method 3)

This notebook trains the Palette colorization model (`configs/m3_palette.yaml`) on the 12,000-image COCO
split, runs Scenario 6 (DDIM sampling steps 50, 100, and 250), and shows the curves and sample colorizations.

**Before running**
1. Select **Runtime → Change runtime type → GPU** (T4 works; L4 or A100 is faster).
2. Make sure the branch in the settings cell is pushed to GitHub.

**If Colab disconnects**: run all cells again from the top. The dataset is restored from Google Drive and
training continues from the last finished epoch (`--resume`).

### Settings

Change these values here only; every later cell uses them.
- `BRANCH`: the Git branch that contains the Palette model.
- `DRIVE_DIR`: Google Drive folder for the dataset zip, checkpoints, and results.
- `SCENARIO6_STEPS`: extra DDIM step counts to test. The 50-step result comes from the training run.

In [ ]:
REPO_URL = "https://github.com/alexcsl/image-recolorization-using-DL.git"
BRANCH = "rev01-palette-diffusion"
DRIVE_DIR = "/content/drive/MyDrive/image-colorization"
REPO_DIR = "/content/image-recolorization-using-DL"
CONFIG = "configs/m3_palette.yaml"
SCENARIO6_STEPS = [100, 250]

## 1. GPU check and Google Drive

Training a diffusion model on a CPU is far too slow, so this stops early if no GPU is attached.
Google Drive is mounted so files survive when the Colab machine is recycled.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch
assert torch.cuda.is_available(), "No GPU found. Select Runtime > Change runtime type > GPU."
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")

## 2. Clone the repository and install dependencies

The repository is cloned to the Colab machine's local disk (faster than Drive). On a re-run in the same
session it pulls the latest commits instead, after undoing the local `num_workers` edit from step 5.
Colab already has PyTorch, OpenCV, and pandas, so only the missing packages are installed.

In [ ]:
import os

if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} checkout -- {CONFIG}
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull --ff-only origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -1
assert os.path.exists("src/models/palette.py"), f"Branch {BRANCH} does not contain the Palette model."

!pip install -q lpips pytorch-fid scikit-image PyYAML

## 3. Keep checkpoints and results on Google Drive

`src.train` writes to `checkpoints/` and `results/` relative to the repository. Both folders are replaced
by links to Drive, so a disconnect does not lose training progress. The first run copies the repository's
existing `results/` (the other experiments' metrics) into Drive.

In [ ]:
import shutil
from pathlib import Path

drive_dir = Path(DRIVE_DIR)
drive_dir.mkdir(parents=True, exist_ok=True)

for name in ["checkpoints", "results"]:
    local = Path(REPO_DIR) / name
    remote = drive_dir / name
    if local.is_symlink():
        continue
    if not remote.exists():
        if local.exists():
            shutil.copytree(local, remote)
        else:
            remote.mkdir(parents=True)
    if local.exists():
        shutil.rmtree(local)
    local.symlink_to(remote, target_is_directory=True)
    print(f"{local} -> {remote}")

## 4. Dataset (12,000 COCO images at 256×256)

The first run downloads the exact images listed in `data/splits/*.csv` (about 15–30 minutes) and saves
them to Drive as one zip. Later runs unzip it from Drive in about a minute.
If a few downloads fail, run the cell again: images already on disk are skipped.

In [ ]:
import pandas as pd

data_zip = drive_dir / "processed_256.zip"
image_dir = Path(REPO_DIR) / "data" / "processed_256"
expected = sum(len(pd.read_csv(f"data/splits/{split}.csv")) for split in ["train", "val", "test"])

def image_count():
    return len(list(image_dir.glob("*.jpg"))) if image_dir.exists() else 0

if image_count() < expected and data_zip.exists():
    !unzip -q -o {data_zip} -d {REPO_DIR}/data

if image_count() < expected:
    !python scripts/prepare_dataset.py --from-splits --workers 32
    if image_count() >= expected:
        !cd {REPO_DIR}/data && zip -q -r -0 /content/processed_256.zip processed_256
        shutil.copy("/content/processed_256.zip", data_zip)
        print("Saved dataset zip to", data_zip)

print(f"Images: {image_count()} / {expected}")
assert image_count() >= expected, "Some images are missing. Run this cell again to retry the download."

## 5. Match the data loader workers to this machine

Colab machines have 2–8 CPU cores. Data loading (JPEG decoding and RGB → Lab conversion) runs on the CPU,
so the worker count is set to the number of cores. This only edits the local copy of the config.

In [ ]:
import yaml

with open(CONFIG, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
cfg["data"]["num_workers"] = min(8, os.cpu_count() or 2)
with open(CONFIG, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("num_workers =", cfg["data"]["num_workers"])
print(yaml.safe_dump({key: cfg[key] for key in ["model", "training"]}, sort_keys=False))

## 6. Sanity check: one training step on the GPU

Runs a single training step on one batch to confirm the model fits in GPU memory at batch size 16 and to
estimate the speed. One epoch is about 600 steps (9,600 training images / 16). Nothing is written to disk.

In [ ]:
import time
from torch.utils.data import DataLoader
from src.dataset import ColorizationDataset
from src.models import build_model
from src.utils import count_parameters

device = torch.device("cuda")
train_ds = ColorizationDataset("data/splits/train.csv", "data/processed_256", 256, augment=True)
L, ab, _ = next(iter(DataLoader(train_ds, batch_size=cfg["training"]["batch_size"], shuffle=True)))
L, ab = L.to(device), ab.to(device)

model = build_model(cfg["model"]).to(device)
print(f"Parameters: {count_parameters(model)[0]:,}")
optimizer = torch.optim.Adam(model.parameters(), lr=float(cfg["training"]["lr"]))

torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
with torch.amp.autocast("cuda"):
    loss = model.diffusion_loss(L, ab)
loss.backward()
optimizer.step()
torch.cuda.synchronize()
print(f"Loss {loss.item():.4f} | step {time.perf_counter() - start:.2f}s | "
      f"peak memory {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

del model, optimizer, loss
torch.cuda.empty_cache()

## 7. Train Palette (30 epochs)

What happens in each epoch:
1. **Training**: for every image a random noise level is chosen, and the U-Net learns to predict the noise
   (L2 loss). An EMA copy of the weights is updated after each step and used for evaluation.
2. **Validation loss** on all 1,200 validation images (same noise every epoch, so epochs are comparable).
3. **PSNR / SSIM / LPIPS** on epochs 1, 5, 10, …, 30: 256 validation images are colorized with 50 DDIM steps.
4. The checkpoint with the lowest validation loss is kept, and the full training state is saved for `--resume`.

After the last epoch the best checkpoint is evaluated on the full test set with PSNR, SSIM, LPIPS, and FID.
FID is computed only on the test set because it needs many images to be reliable.
If the session disconnects, run all cells from the top; this cell continues from `checkpoints/m3_palette_last.pt`.

In [ ]:
!python -m src.train --config {CONFIG} --resume

## 8. Scenario 6: number of DDIM sampling steps

More sampling steps usually give slightly better colors but make inference proportionally slower.
The trained checkpoint is reused, and only the number of steps changes at test time. Each run adds a row
such as `m3_palette_ddim100` to `results/metrics_summary.csv`.

In [ ]:
for steps in SCENARIO6_STEPS:
    !python -m src.train --config {CONFIG} --eval-only --sampling-steps {steps}

## 9. Results

### Test-set metrics

Compare the Palette rows (50, 100, and 250 DDIM steps) with each other and with the other models.
PSNR and SSIM are higher-is-better; LPIPS and FID are lower-is-better. A diffusion model often has a
lower PSNR than a regression model (its colors are sharper but not always the original ones) and a better
FID (its colors look more like real photos). `test_time_sec` shows the inference cost of more sampling steps.

In [ ]:
summary = pd.read_csv("results/metrics_summary.csv")
columns = ["experiment_name", "model", "sampling_steps", "epochs", "best_epoch",
           "test_psnr", "test_ssim", "test_lpips", "test_fid", "train_time_min", "test_time_sec"]
display(summary[[c for c in columns if c in summary.columns]])

### Training curves

- **Denoising loss**: the training objective. Train and validation should fall together; a growing gap
  means overfitting.
- **PSNR / SSIM / LPIPS**: quality of the actual colorizations on validation images (solid) and on a fixed
  sample of training images (dashed). The figure is also saved to `results/plots/`.

In [ ]:
import matplotlib.pyplot as plt

exp = cfg["experiment_name"]
history = pd.read_csv(f"results/history/{exp}_history.csv")
metric_history = history.dropna(subset=["val_psnr"])

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
axes[0].plot(history["epoch"], history["train_loss"], label="Train")
axes[0].plot(history["epoch"], history["val_loss"], "--", label="Val")
axes[0].set_title("Denoising loss (L2 on noise)")
for ax, metric, title in zip(axes[1:], ["psnr", "ssim", "lpips"], ["PSNR (dB) ↑", "SSIM ↑", "LPIPS ↓"]):
    ax.plot(metric_history["epoch"], metric_history[f"val_{metric}"], "o-", label="Val")
    ax.plot(metric_history["epoch"], metric_history[f"train_{metric}"], ".--", label="Train sample")
    ax.set_title(title)
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3)
    ax.legend()
plt.suptitle("Palette training curves")
plt.tight_layout()
Path("results/plots").mkdir(parents=True, exist_ok=True)
plt.savefig(f"results/plots/{exp}_curves.png", dpi=160)
plt.show()

### Sample colorizations

Six evenly spaced test images colorized by the best checkpoint with 50 DDIM steps. `colorize` (from
`src.evaluate`) fixes the random seed, so the colors match the ones scored in the test evaluation.
The figure is also saved to `results/samples/`.

In [ ]:
import numpy as np
from src.evaluate import colorize, load_trained_model
from src.utils import lab_normalized_to_rgb

model = load_trained_model(cfg, device)
test_ds = ColorizationDataset("data/splits/test.csv", "data/processed_256", 256)
indices = np.linspace(0, len(test_ds) - 1, 6, dtype=int)
L = torch.stack([test_ds[i][0] for i in indices]).to(device)
ab = torch.stack([test_ds[i][1] for i in indices]).to(device)

with torch.amp.autocast("cuda"):
    pred_ab = colorize(model, L, seed=cfg["seed"])

columns = [
    (((L[:, 0].cpu().numpy() + 1) / 2).clip(0, 1), "Grayscale (L)"),
    (lab_normalized_to_rgb(L, pred_ab), "Palette"),
    (lab_normalized_to_rgb(L, ab), "Ground truth"),
]
fig, axes = plt.subplots(len(indices), 3, figsize=(9, 3 * len(indices)))
for row in range(len(indices)):
    for col, (images, title) in enumerate(columns):
        axes[row, col].imshow(images[row], cmap="gray" if col == 0 else None, vmin=0, vmax=1)
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(title)
plt.tight_layout()
Path("results/samples").mkdir(parents=True, exist_ok=True)
plt.savefig(f"results/samples/{exp}_samples.png", dpi=160, bbox_inches="tight")
plt.show()

## 10. Package the results for the repository

Creates `palette_results.zip` in Drive with the training history, summary JSON files, metrics table,
curves, and samples. Unzip it at the repository root and commit it. Checkpoints stay in `checkpoints/`
on Drive and are not committed (they are too large for Git).

In [ ]:
import zipfile

files = sorted(Path("results/history").glob(f"{exp}*")) + [
    Path("results/metrics_summary.csv"),
    Path(f"results/plots/{exp}_curves.png"),
    Path(f"results/samples/{exp}_samples.png"),
]
out_path = drive_dir / "palette_results.zip"
with zipfile.ZipFile(out_path, "w") as zf:
    for file in files:
        if file.exists():
            zf.write(file, file.as_posix())
            print("added", file)
print("Saved", out_path)